# Lab 4.4 — Tri-mode `generate()`

This lab unifies AR, block-diffusion, and self-speculation into a single `generate(mode=...)` function. The model from Lab 4.3 already supports both AR and diffusion modes; we now expose them through one entrypoint and add a comparison harness.

**Goals.**
1. Implement a `generate(prompt, mode, ...)` that dispatches on three modes.
2. Compare quality and TPF across modes on the same prompts.
3. Show that **the same weight matrix** handles all three modes — the only thing that changes is the attention mask and the decoding loop.

**Prerequisites.** Labs 4.1–4.3. Lectures 2.3, 3.2.


In [1]:
# Cell 1 — boilerplate (same as Lab 4.3, abbreviated)
import math, time
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

text = (
    "The quick brown fox jumps over the lazy dog. "
    "Pack my box with five dozen liquor jugs. "
    "How vexingly quick daft zebras jump! "
    "Sphinx of black quartz, judge my vow. "
    "The five boxing wizards jump quickly. "
) * 2000
chars = sorted(list(set(text))) + ["<MASK>"]
vocab_size = len(chars)
mask_token_id = vocab_size - 1
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: "".join(itos[i] if i != mask_token_id else "_" for i in l)
data = torch.tensor(encode(text), dtype=torch.long)
n_train = int(0.9 * len(data))
train_data = data[:n_train]


/home/ubuntu/.pyenv/versions/3.12.8/lib/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [2]:
# Cell 2 — model from Lab 4.3
@dataclass
class GPTConfig:
    vocab_size: int = 35
    n_layer: int = 4
    n_head: int = 4
    n_embd: int = 128
    block_size: int = 256
    diff_block_size: int = 8

class FlexAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_head, self.n_embd = config.n_head, config.n_embd
        self.qkv = nn.Linear(config.n_embd, 3 * config.n_embd, bias=False)
        self.proj = nn.Linear(config.n_embd, config.n_embd, bias=False)
    def forward(self, x, attn_mask):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(self.n_embd, dim=2)
        head_dim = C // self.n_head
        q = q.view(B, T, self.n_head, head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, head_dim).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(head_dim)
        att = att.masked_fill(~attn_mask.unsqueeze(0).unsqueeze(0), float("-inf"))
        att = F.softmax(att, dim=-1)
        return self.proj((att @ v).transpose(1, 2).contiguous().view(B, T, C))

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=False)
        self.proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=False)
    def forward(self, x):
        return self.proj(F.gelu(self.fc(x)))

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = nn.LayerNorm(config.n_embd)
        self.attn = FlexAttention(config)
        self.ln2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)
    def forward(self, x, attn_mask):
        x = x + self.attn(self.ln1(x), attn_mask)
        return x + self.mlp(self.ln2(x))

class TinyJointModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.tok_emb = nn.Embedding(config.vocab_size, config.n_embd)
        self.pos_emb = nn.Embedding(config.block_size, config.n_embd)
        self.blocks = nn.ModuleList([Block(config) for _ in range(config.n_layer)])
        self.ln_f = nn.LayerNorm(config.n_embd)
        self.head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
    def forward(self, idx, attn_mask):
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(idx.shape[1], device=idx.device))
        for blk in self.blocks:
            x = blk(x, attn_mask)
        return self.head(self.ln_f(x))

def make_causal_mask(L, device):
    return torch.tril(torch.ones(L, L, dtype=torch.bool, device=device))

def make_dual_stream_mask(L, block_size, device):
    total = 2 * L
    q = torch.arange(total, device=device).view(-1, 1)
    kv = torch.arange(total, device=device).view(1, -1)
    x0_flag_q = (q >= L).long()
    x0_flag_kv = (kv >= L).long()
    block_q = torch.where(x0_flag_q == 1, (q - L) // block_size, q // block_size)
    block_kv = torch.where(x0_flag_kv == 1, (kv - L) // block_size, kv // block_size)
    M_BD = (block_q == block_kv) & (x0_flag_q == x0_flag_kv)
    M_OBC = (block_q > block_kv) & (x0_flag_kv == 1) & (x0_flag_q == 0)
    M_BC = (block_q >= block_kv) & (x0_flag_kv == 1) & (x0_flag_q == 1)
    return M_BD | M_OBC | M_BC

config = GPTConfig(vocab_size=vocab_size, n_layer=4, n_head=4, n_embd=128,
                    block_size=256, diff_block_size=8)
model = TinyJointModel(config).to(device)

# Quick joint training (re-use from Lab 4.3)
def get_batch(batch_size=16, seq_len=64):
    ix = torch.randint(len(train_data) - seq_len, (batch_size,))
    return torch.stack([train_data[i:i+seq_len] for i in ix]).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)
print("Training (150 steps)...")
for step in range(150):
    x = get_batch()
    B, L = x.shape
    rho = 0.01 + 0.98 * torch.rand(B, 1, device=device)
    mask_indices = torch.rand(B, L, device=device) < rho
    if not mask_indices.any():
        mask_indices[0, 0] = True
    noisy = torch.where(mask_indices, mask_token_id, x)
    # AR loss
    causal = make_causal_mask(L, device)
    ar_logits = model(x, causal)
    L_ar = F.cross_entropy(ar_logits[:, :-1].reshape(-1, ar_logits.size(-1)),
                            x[:, 1:].reshape(-1))
    # Diffusion loss
    dual = torch.cat([noisy, x], dim=-1)
    bd_mask = make_dual_stream_mask(L, config.diff_block_size, device)
    bd_logits = model(dual, bd_mask)
    noisy_logits = bd_logits[:, :L]
    L_diff = F.cross_entropy(noisy_logits[mask_indices].view(-1, noisy_logits.size(-1)),
                              x[mask_indices].view(-1))
    loss = L_ar + 0.5 * L_diff
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    if step % 50 == 0 or step == 149:
        print(f"  step {step}: L_ar {L_ar.item():.3f}  L_diff {L_diff.item():.3f}")
print("Training done.")


Training (150 steps)...
  step 0: L_ar 3.783  L_diff 3.806


  step 50: L_ar 0.311  L_diff 3.098


  step 100: L_ar 0.075  L_diff 2.445


  step 149: L_ar 0.055  L_diff 1.521
Training done.


## 1. The unified `generate(mode=...)` entrypoint


In [3]:
# Cell 3 — three generate functions
@torch.no_grad()
def generate_ar(model, prompt: str, max_new_tokens: int = 50):
    model.eval()
    ids = torch.tensor(encode(prompt), dtype=torch.long, device=device).unsqueeze(0)
    nfe = 0
    for _ in range(max_new_tokens):
        idx = ids[:, -config.block_size:]
        causal = make_causal_mask(idx.shape[1], device)
        logits = model(idx, causal)
        nfe += 1
        next_id = logits[0, -1].argmax().unsqueeze(0).unsqueeze(0)
        ids = torch.cat([ids, next_id], dim=1)
    return decode(ids[0].tolist()), nfe

@torch.no_grad()
def generate_block_diffusion(model, prompt: str, gen_length: int = 32,
                                block_length: int = 8, threshold: float = 0.3):
    model.eval()
    prompt_ids = torch.tensor(encode(prompt), dtype=torch.long, device=device)
    committed = prompt_ids.clone()
    nfe = 0
    remaining = gen_length
    while remaining > 0:
        block = min(block_length, remaining)
        L = committed.shape[0] + block
        noisy = torch.cat([committed,
                            torch.full((block,), mask_token_id, device=device)])
        dual = torch.cat([noisy, noisy]).unsqueeze(0)
        mask = make_dual_stream_mask(L, block_length, device)
        logits = model(dual, mask)
        nfe += 1
        probs = F.softmax(logits[0, :L], dim=-1)
        block_probs = probs[committed.shape[0]:committed.shape[0] + block]
        max_p, argmax = block_probs.max(dim=-1)
        # Threshold accept (with at least one)
        n_accept = 1
        for i in range(block):
            if max_p[i] >= threshold:
                n_accept = i + 1
            else:
                break
        if n_accept == 0:
            n_accept = 1
        committed = torch.cat([committed, argmax[:n_accept]])
        remaining -= n_accept
    return decode(committed.tolist()), nfe

@torch.no_grad()
def generate_self_speculation(model, prompt: str, gen_length: int = 32,
                                  block_length: int = 8, threshold: float = 0.3):
    '''Draft with diffusion, verify with AR. Both use the same weights.'''
    model.eval()
    prompt_ids = torch.tensor(encode(prompt), dtype=torch.long, device=device)
    committed = prompt_ids.clone()
    nfe = 0
    remaining = gen_length
    while remaining > 0:
        block = min(block_length, remaining)
        L = committed.shape[0] + block
        
        # 1. Diffusion DRAFT
        noisy = torch.cat([committed,
                            torch.full((block,), mask_token_id, device=device)])
        dual = torch.cat([noisy, noisy]).unsqueeze(0)
        bd_mask = make_dual_stream_mask(L, block_length, device)
        bd_logits = model(dual, bd_mask)
        nfe += 1
        bd_probs = F.softmax(bd_logits[0, :L], dim=-1)
        block_probs = bd_probs[committed.shape[0]:committed.shape[0] + block]
        draft_max, draft_argmax = block_probs.max(dim=-1)
        
        # 2. AR VERIFY (causal forward with drafts as input)
        verify_input = torch.cat([committed, draft_argmax]).unsqueeze(0)
        causal = make_causal_mask(L, device)
        verify_logits = model(verify_input, causal)
        nfe += 1
        verify_argmax = verify_logits[0, committed.shape[0]-1:L-1].argmax(dim=-1)
        # verify_argmax[i] = AR prediction at position committed.shape[0] + i
        # i.e., predicting what should follow the previous (committed or drafted) token.
        
        # 3. ACCEPT longest prefix that matches AND has high confidence
        accept = (draft_argmax == verify_argmax) & (draft_max >= threshold)
        n_accept = 0
        for i in range(block):
            if accept[i]:
                n_accept += 1
            else:
                break
        if n_accept == 0:
            n_accept = 1  # always commit at least the first AR-argmax token
            committed = torch.cat([committed, verify_argmax[:1]])
        else:
            committed = torch.cat([committed, draft_argmax[:n_accept]])
        remaining -= n_accept
    return decode(committed.tolist()), nfe

def generate(model, prompt, mode="ar", **kwargs):
    if mode == "ar":
        return generate_ar(model, prompt, **kwargs)
    elif mode == "block_diffusion":
        return generate_block_diffusion(model, prompt, **kwargs)
    elif mode == "self_speculation":
        return generate_self_speculation(model, prompt, **kwargs)
    else:
        raise ValueError(f"Unknown mode: {mode}")


## 2. Side-by-side comparison


In [4]:
# Cell 4 — three modes on the same prompt
prompts = ["The quick", "Pack my box", "Sphinx of"]

print(f"{'mode':<20} {'tokens committed':<20} {'NFE':<8} {'TPF':<8}")
print("-" * 60)
for prompt in prompts:
    print(f"\nPrompt: {prompt!r}")
    for mode in ["ar", "block_diffusion", "self_speculation"]:
        gen_kwargs = {"max_new_tokens": 32} if mode == "ar" else {"gen_length": 32}
        text, nfe = generate(model, prompt, mode=mode, **gen_kwargs)
        n_new = len(text) - len(prompt)
        tpf = n_new / nfe if nfe > 0 else 0
        print(f"  {mode:<20} {n_new:<20} {nfe:<8} {tpf:.2f}")


mode                 tokens committed     NFE      TPF     
------------------------------------------------------------

Prompt: 'The quick'
  ar                   32                   32       1.00
  block_diffusion      32                   5        6.40


  self_speculation     32                   64       0.50

Prompt: 'Pack my box'
  ar                   32                   32       1.00
  block_diffusion      32                   5        6.40


  self_speculation     32                   64       0.50

Prompt: 'Sphinx of'
  ar                   32                   32       1.00
  block_diffusion      32                   11       2.91


  self_speculation     32                   62       0.52


## 3. Wall-clock measurement

The TPF tells us tokens per forward, but wall-clock depends on the per-forward cost. Self-spec uses 2 forwards per cycle (one dual-stream of length 2L, one causal of length L), so its per-forward wall-clock is heavier.


In [5]:
# Cell 5 — wall-clock per mode
def time_generate(generate_fn, mode, n_runs=3, **kwargs):
    times = []
    total_new = 0
    for _ in range(n_runs):
        t0 = time.time()
        text, nfe = generate_fn(model, "The quick", mode=mode, **kwargs)
        times.append(time.time() - t0)
        n_new = len(text) - len("The quick")
        total_new += n_new
    avg_t = sum(times) / len(times)
    avg_new = total_new / n_runs
    return avg_t, avg_new

print(f"{'mode':<20} {'wall_clock (s)':<18} {'tokens/run':<12} {'tokens/s':<10}")
print("-" * 60)
for mode in ["ar", "block_diffusion", "self_speculation"]:
    kwargs = {"max_new_tokens": 50} if mode == "ar" else {"gen_length": 50}
    t, new = time_generate(generate, mode, **kwargs)
    print(f"{mode:<20} {t:<18.3f} {new:<12.1f} {new/t:<10.1f}")


mode                 wall_clock (s)     tokens/run   tokens/s  
------------------------------------------------------------


ar                   0.129              50.0         386.1     
block_diffusion      0.033              50.0         1509.3    


self_speculation     0.301              50.0         166.2     


## 4. Sanity checks


In [6]:
# Cell 6 — assertions
# Modes are dispatched correctly
for mode in ["ar", "block_diffusion", "self_speculation"]:
    text, nfe = generate(model, "The", mode=mode,
                          **({"max_new_tokens": 10} if mode == "ar" else {"gen_length": 10}))
    assert nfe > 0
    assert len(text) >= len("The")

# Invalid mode raises
try:
    generate(model, "X", mode="invalid")
    assert False, "should have raised"
except ValueError:
    pass

print("All assertions passed.")


All assertions passed.


## 5. Recap

We have a unified `generate(mode=...)` that:
- Supports **3 modes** with the same weight matrix.
- AR mode: one forward per token, TPF = 1.
- Block-diffusion: one forward per block, TPF ≈ 5–8 (block_length / cycles).
- Self-speculation: two forwards per block (draft + verify), TPF ≈ 4–6.

Self-spec is slower-per-cycle than pure block-diffusion but its acceptance check guards against quality drops. Lab 4.5 implements the **KV cache reuse** that makes self-spec actually faster in wall-clock.

Next lab (4.5): implement KV-cache sharing between the draft and verify forwards.
